# S4b.1 — Cross-Encoder Re-ranking

Interactive verification of the re-ranking service.

- **FR-1**: RerankerService interface (`rerank()`)
- **FR-2**: Local cross-encoder scoring (sigmoid normalization)
- **FR-3**: Cohere provider (optional)
- **FR-4**: Integration with `SearchHit` (`rerank_search_hits()`)
- **FR-5**: Configuration via `RerankerSettings`
- **FR-6**: Factory function + DI

In [1]:
import sys
sys.path.insert(0, "../..")

from unittest.mock import MagicMock
import numpy as np

print("\u2713 Imports ready")

✓ Imports ready


## FR-5: Configuration

In [2]:
from src.config import RerankerSettings

settings = RerankerSettings()
print(f"Model: {settings.model}")
print(f"Top-K: {settings.top_k}")
print(f"Device: {settings.device}")
print(f"Provider: {settings.provider}")
assert settings.model == "cross-encoder/ms-marco-MiniLM-L-12-v2"
assert settings.top_k == 5
assert settings.provider == "local"
print("\n\u2713 RerankerSettings configured correctly")

Model: cross-encoder/ms-marco-MiniLM-L-12-v2
Top-K: 5
Device: cpu
Provider: local

✓ RerankerSettings configured correctly


## FR-1: RerankerService Interface

In [3]:
from src.services.reranking.service import RerankerService, RerankResult

# Create service with mocked model
mock_model = MagicMock()
mock_model.predict.return_value = np.array([0.9, -0.5, 0.2, 1.5, -1.0])
service = RerankerService(settings=settings, model=mock_model)

documents = [
    {"text": "Transformers use self-attention.", "id": "doc1"},
    {"text": "CNNs for image classification.", "id": "doc2"},
    {"text": "RNNs process sequences.", "id": "doc3"},
    {"text": "BERT is bidirectional.", "id": "doc4"},
    {"text": "GPT generates text.", "id": "doc5"},
]

results = await service.rerank("transformer architecture", documents)

print(f"Top-{len(results)} results:")
for r in results:
    print(f"  idx={r.index}, score={r.relevance_score:.4f}, doc={r.document['id']}")

# Verify sorted descending
scores = [r.relevance_score for r in results]
assert scores == sorted(scores, reverse=True)
assert len(results) == settings.top_k
print(f"\n\u2713 rerank() returns {len(results)} sorted results")

Top-5 results:
  idx=3, score=0.8176, doc=doc4
  idx=0, score=0.7109, doc=doc1
  idx=2, score=0.5498, doc=doc3
  idx=1, score=0.3775, doc=doc2
  idx=4, score=0.2689, doc=doc5

✓ rerank() returns 5 sorted results


## FR-2: Score Normalization

In [4]:
from src.services.reranking.service import _sigmoid

raw = np.array([-5.0, -1.0, 0.0, 1.0, 5.0])
normalized = _sigmoid(raw)
print(f"Raw scores:        {raw}")
print(f"Normalized (0-1):  {normalized}")

assert all(0.0 <= s <= 1.0 for s in normalized)
assert abs(normalized[2] - 0.5) < 1e-6  # sigmoid(0) = 0.5
print("\n\u2713 Scores normalized to 0.0-1.0 via sigmoid")

Raw scores:        [-5. -1.  0.  1.  5.]
Normalized (0-1):  [0.00669285 0.26894142 0.5        0.73105858 0.99330715]

✓ Scores normalized to 0.0-1.0 via sigmoid


## FR-4: SearchHit Re-ranking

In [5]:
from src.schemas.api.search import SearchHit

hits = [
    SearchHit(arxiv_id="2301.001", title="Paper A", chunk_text="Attention mechanisms.", score=0.8),
    SearchHit(arxiv_id="2301.002", title="Paper B", chunk_text="Image recognition.", score=0.7),
    SearchHit(arxiv_id="2301.003", title="Paper C", chunk_text="", abstract="Sequence models.", score=0.6),
    SearchHit(arxiv_id="2301.004", title="Paper D", chunk_text="BERT pretraining.", score=0.5),
]

mock_model2 = MagicMock()
mock_model2.predict.return_value = np.array([2.0, -1.0, 0.5, 1.0])
service2 = RerankerService(settings=settings, model=mock_model2)

reranked = await service2.rerank_search_hits("attention transformers", hits, top_k=3)

print("Re-ranked SearchHits:")
for h in reranked:
    print(f"  {h.arxiv_id} | {h.title} | score={h.score:.4f}")

assert len(reranked) == 3
assert all(isinstance(h, SearchHit) for h in reranked)
scores = [h.score for h in reranked]
assert scores == sorted(scores, reverse=True)
print("\n\u2713 rerank_search_hits() re-orders and updates scores")

Re-ranked SearchHits:
  2301.001 | Paper A | score=0.8808
  2301.004 | Paper D | score=0.7311
  2301.003 | Paper C | score=0.6225

✓ rerank_search_hits() re-orders and updates scores


## FR-6: Factory & DI

In [6]:
from unittest.mock import patch

with patch("src.services.reranking.factory.CrossEncoder") as mock_ce:
    mock_ce.return_value = MagicMock()
    from src.services.reranking.factory import create_reranker_service
    svc = create_reranker_service(settings)
    print(f"Service created: {type(svc).__name__}")
    print(f"Provider: {svc._provider}")
    assert isinstance(svc, RerankerService)

from src.dependency import RerankerDep
print(f"RerankerDep available: {RerankerDep}")

print("\n\u2713 Factory + DI wired correctly")

Service created: RerankerService
Provider: local
RerankerDep available: typing.Annotated[src.services.reranking.service.RerankerService, Depends(dependency=<function get_reranker_service at 0x165717920>, use_cache=True, scope=None)]

✓ Factory + DI wired correctly


## Edge Cases

In [7]:
# Empty input
empty_result = await service.rerank("query", [])
assert empty_result == []
print("\u2713 Empty documents \u2192 empty result")

# top_k > docs
mock_model3 = MagicMock()
mock_model3.predict.return_value = np.array([0.5, 0.9])
svc3 = RerankerService(settings=settings, model=mock_model3)
r = await svc3.rerank("q", [{"text": "a", "id": 1}, {"text": "b", "id": 2}], top_k=100)
assert len(r) == 2
print("\u2713 top_k > docs \u2192 returns all docs")

print("\n\u2713 All edge cases pass")

✓ Empty documents → empty result
✓ top_k > docs → returns all docs

✓ All edge cases pass
